# Multi-Task Fine-Tuning of Foundational Models for Retinal Disease Detection

Fine-tunes **all 7 foundational model backbones** using LoRA with **multi-task learning**.
The backbone learns to detect **multiple conditions simultaneously**, producing embeddings
that generalize well across all downstream tasks.

### Why multi-task?
Single-task fine-tuning causes **catastrophic forgetting** — a model tuned for DR
loses the ability to detect glaucoma. Multi-task forces the shared backbone to learn
features useful for ALL conditions: DR, glaucoma, macular edema, drusens, etc.

### Tasks trained simultaneously (BRSET — 16,266 images)
| Task | Positive Rate | What it detects |
|---|---|---|
| `diabetic_retinopathy` | 6.4% | Any DR |
| `increased_cup_disc` | 19.7% | Glaucoma suspect |
| `macular_edema` | 2.5% | Macular edema |
| `drusens` | 17.3% | Drusen deposits |
| `hypertensive_retinopathy` | 1.7% | Hypertensive changes |
| `myopic_fundus` | 1.6% | Myopic degeneration |
| `amd` | 2.3% | Age-related macular degeneration |
| `scar` | 1.8% | Retinal scarring |

### Backbones (all 7)
1. `convnextv2_base` — ConvNeXt V2 Base (ImageNet-22K)
2. `dinov3_convnext_base` — DINOv3 ConvNeXt Base (LVD-142M)
3. `dinov3_vitb16` — DINOv3 ViT-B/16 (LVD-142M)
4. `RETFound_dinov2_shanghai` — RETFound DINOv2 (Shanghai Ophthalmic)
5. `RETFound_mae_natureCFP` — RETFound MAE (Natural CFP)
6. `RETFound_mae_shanghai` — RETFound MAE (Shanghai Ophthalmic)
7. `vit_base` — ViT-B/16 (ImageNet-21K)

### Output
General-purpose fine-tuned embeddings in the **same CSV format** as frozen embeddings,
directly usable in existing evaluation notebooks.

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q torch torchvision
!pip install -q "transformers>=4.44.0" "huggingface_hub>=0.23.0"
!pip install -q "peft>=0.12.0"
!pip install -q scikit-learn pandas numpy tqdm

# For RETFound models — clone their repo
import os
if not os.path.exists('/content/RETFound'):
    !git clone https://github.com/rmaphoh/RETFound.git /content/RETFound
    %cd /content/RETFound
    !pip install -q -r requirements.txt
    %cd /content

print('All dependencies installed.')

In [ ]:
# ============================================================
# Cell 2: Authentication & Colab Setup
# ============================================================
import os
import getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Enter your Hugging Face token: ')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(HF_TOKEN, add_to_git_credential=False)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. Fine-tuning will be very slow.')
print(f'Device: {device}')

In [ ]:
# ============================================================
# Cell 3: CONFIGURATION
# ============================================================

# ---- Dataset ----
# Use BRSET (16,266 images) for multi-task — it has all the pathology labels
DATASET = 'brset'

# ---- Paths (ADJUST THESE for your Drive) ----
IMAGE_DIR = '/content/drive/MyDrive/Image_data/Data/brset/images_224'
LABELS_CSV = '/content/drive/MyDrive/Image_data/Data/brset/labels_brset.csv'
OUTPUT_DIR = '/content/drive/MyDrive/finetuned_embeddings_multitask'

# ---- Tasks for multi-task learning ----
# These are ALL viable binary tasks from BRSET labels.
# Each has enough positives (>90) to train on.
TASKS = [
    'diabetic_retinopathy',     # DR detection (6.4%)
    'increased_cup_disc',       # Glaucoma suspect (19.7%)
    'macular_edema',            # Macular edema (2.5%)
    'drusens',                  # Drusen deposits (17.3%)
    'hypertensive_retinopathy', # Hypertensive changes (1.7%)
    'myopic_fundus',            # Myopic degeneration (1.6%)
    'amd',                      # AMD (2.3%)
    'scar',                     # Retinal scarring (1.8%)
]

# ---- Task weights ----
# Controls how much each task influences the shared backbone.
# Higher weight = more influence. Rare conditions get higher weight
# so they don't get drowned out by common ones.
TASK_LOSS_WEIGHTS = {
    'diabetic_retinopathy':     1.5,  # Primary task (DR)
    'increased_cup_disc':       1.5,  # Primary task (glaucoma)
    'macular_edema':            1.2,  # Important for referral
    'drusens':                  0.8,  # Common, less weight needed
    'hypertensive_retinopathy': 1.0,
    'myopic_fundus':            1.0,
    'amd':                      1.0,
    'scar':                     0.8,
}

# ---- Training hyperparameters ----
LORA_RANK = 16          # LoRA rank
LORA_ALPHA = 32         # LoRA alpha
LR_HEAD = 1e-3          # Learning rate for classification heads
LR_BACKBONE = 2e-5      # Learning rate for LoRA backbone params
BATCH_SIZE = 32         # Reduce to 16 if OOM
EPOCHS = 30
PATIENCE = 7            # Early stopping patience
WEIGHT_DECAY = 0.01
NUM_WORKERS = 2
VAL_FRACTION = 0.15

print(f'Dataset:    {DATASET}')
print(f'Image dir:  {IMAGE_DIR}')
print(f'Tasks ({len(TASKS)}): {TASKS}')
print(f'Output:     {OUTPUT_DIR}')

In [ ]:
# ============================================================
# Cell 4: Load Labels & Patient-Level Split
# ============================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

labels_raw = pd.read_csv(LABELS_CSV)
print(f'Loaded {len(labels_raw)} rows from {LABELS_CSV}')

# --- Prepare labels ---
labels = labels_raw.copy()
labels['image_file'] = labels['image_id'].astype(str) + '.jpg'

# Coerce task columns to binary (0/1/NaN)
for task in TASKS:
    labels[task] = pd.to_numeric(labels[task], errors='coerce')
    # Keep only 0 and 1, everything else becomes NaN
    labels[task] = labels[task].where(labels[task].isin([0, 1]))

# --- Verify images exist on disk ---
existing_files = set(os.listdir(IMAGE_DIR))
labels = labels[labels['image_file'].isin(existing_files)].copy()
print(f'Images found on disk: {len(labels)}')

# --- Task label coverage ---
print(f'\nTask label coverage:')
pos_weights = {}  # Per-task class weights for BCE
for task in TASKS:
    valid = labels[task].notna().sum()
    pos = (labels[task] == 1).sum()
    neg = (labels[task] == 0).sum()
    rate = pos / max(valid, 1)
    pw = neg / max(pos, 1)  # pos_weight = n_neg / n_pos
    pos_weights[task] = min(pw, 20.0)  # Cap at 20 to avoid instability
    print(f'  {task:30s}  valid={valid:6d}  pos={pos:5d}  neg={neg:5d}  '
          f'rate={rate:.3f}  pos_weight={pos_weights[task]:.1f}')

# --- Patient-level train/test split (80/20) ---
# Stratify on the most imbalanced primary task (DR has only 6.4%)
patient_dr = labels.groupby('patient_id')['diabetic_retinopathy'].max().fillna(0).astype(int)

train_pats, test_pats = train_test_split(
    patient_dr.index.tolist(),
    test_size=0.20,
    random_state=42,
    stratify=patient_dr.values
)

# Further split train → train/val
train_dr = patient_dr.loc[train_pats]
train_pats_final, val_pats = train_test_split(
    train_pats,
    test_size=VAL_FRACTION,
    random_state=42,
    stratify=train_dr.values
)

train_df = labels[labels['patient_id'].isin(train_pats_final)].reset_index(drop=True)
val_df = labels[labels['patient_id'].isin(val_pats)].reset_index(drop=True)
test_df = labels[labels['patient_id'].isin(test_pats)].reset_index(drop=True)

print(f'\nPatient-level split:')
print(f'  Train: {len(train_pats_final)} patients, {len(train_df)} images')
print(f'  Val:   {len(val_pats)} patients, {len(val_df)} images')
print(f'  Test:  {len(test_pats)} patients, {len(test_df)} images')

# Save patient split for reproducibility
os.makedirs(OUTPUT_DIR, exist_ok=True)
split_records = (
    [{'patient_id': p, 'split': 'train'} for p in train_pats_final] +
    [{'patient_id': p, 'split': 'val'} for p in val_pats] +
    [{'patient_id': p, 'split': 'test'} for p in test_pats]
)
pd.DataFrame(split_records).to_csv(
    os.path.join(OUTPUT_DIR, 'patient_split_multitask.csv'), index=False
)
print('Patient split saved.')

In [ ]:
# ============================================================
# Cell 5: Multi-Task Dataset with Augmentation
# ============================================================
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


class MultiTaskRetinaDataset(Dataset):
    """
    Returns (image, label_dict, mask_dict, filename).
    - label_dict: {task_name: 0 or 1} for each task
    - mask_dict:  {task_name: 1 if label is available, 0 if NaN}
    Missing labels get mask=0, so their loss is zeroed out.
    """

    def __init__(self, df, image_dir, tasks, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.tasks = tasks
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['image_file'])
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        # Build label and mask tensors
        labels = torch.zeros(len(self.tasks), dtype=torch.float32)
        masks = torch.zeros(len(self.tasks), dtype=torch.float32)

        for i, task in enumerate(self.tasks):
            val = row[task]
            if pd.notna(val):
                labels[i] = float(val)
                masks[i] = 1.0  # This task has a valid label
            # else: label=0, mask=0 → loss will be zeroed

        return img, labels, masks, row['image_file']


# --- Transforms ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- DataLoaders ---
train_dataset = MultiTaskRetinaDataset(train_df, IMAGE_DIR, TASKS, train_transform)
val_dataset = MultiTaskRetinaDataset(val_df, IMAGE_DIR, TASKS, eval_transform)
test_dataset = MultiTaskRetinaDataset(test_df, IMAGE_DIR, TASKS, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# Full dataset (all images, no augmentation) for embedding extraction
full_dataset = MultiTaskRetinaDataset(labels, IMAGE_DIR, TASKS, eval_transform)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} images, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} images, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} images, {len(test_loader)} batches')
print(f'Full:  {len(full_dataset)} images (for embedding extraction)')

In [ ]:
# ============================================================
# Cell 6: Backbone Loader (all 7 models)
# ============================================================
import torch.nn as nn
from transformers import (
    AutoModel, AutoImageProcessor,
    ConvNextV2Model,
    ViTModel,
)

import sys
sys.path.insert(0, '/content/RETFound')


def load_backbone(backbone_name):
    """
    Load backbone and return (model, embed_dim, lora_target_modules).
    """
    if backbone_name == 'convnextv2_base':
        model = ConvNextV2Model.from_pretrained('facebook/convnextv2-base-22k-224')
        return model, 1024, ['pwconv1', 'pwconv2']

    elif backbone_name == 'dinov3_convnext_base':
        model = AutoModel.from_pretrained(
            'facebook/dinov3-convnext-base-pretrain-lvd1689m', token=HF_TOKEN
        )
        targets = _find_lora_targets(model, prefer=['pwconv1', 'pwconv2', 'dense'])
        return model, 1024, targets

    elif backbone_name == 'dinov3_vitb16':
        model = AutoModel.from_pretrained(
            'facebook/dinov3-vitb16-pretrain-lvd1689m', token=HF_TOKEN
        )
        return model, 768, ['query', 'value']

    elif backbone_name == 'RETFound_dinov2_shanghai':
        from huggingface_hub import hf_hub_download
        import models_vit as retfound_models
        chkpt = hf_hub_download(
            repo_id='YukunZhou/RETFound_dinov2_shanghai',
            filename='RETFound_dinov2_shanghai.pth', token=HF_TOKEN
        )
        class _Args:
            model_arch = 'retfound_dinov2'
            nb_classes = 5
        model = retfound_models.RETFound_dinov2(_Args(), num_classes=5)
        ckpt = torch.load(chkpt, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['teacher'], strict=False)
        targets = _find_lora_targets(model, prefer=['qkv', 'proj'])
        return model, 1024, targets

    elif backbone_name in ('RETFound_mae_natureCFP', 'RETFound_mae_shanghai'):
        from huggingface_hub import hf_hub_download
        import models_vit as retfound_models
        chkpt = hf_hub_download(
            repo_id=f'YukunZhou/{backbone_name}',
            filename=f'{backbone_name}.pth', token=HF_TOKEN
        )
        model = retfound_models.__dict__['RETFound_mae'](
            img_size=224, num_classes=5, drop_path_rate=0, global_pool=True
        )
        ckpt = torch.load(chkpt, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['model'], strict=False)
        targets = _find_lora_targets(model, prefer=['qkv', 'proj'])
        return model, 1024, targets

    elif backbone_name == 'vit_base':
        model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        return model, 768, ['query', 'value']

    else:
        raise ValueError(f'Unknown backbone: {backbone_name}')


def _find_lora_targets(model, prefer=None):
    """Auto-detect LoRA target module names from Linear layers."""
    prefer = prefer or []
    all_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            all_names.add(name.split('.')[-1])

    found = [p for p in prefer if p in all_names]
    if found:
        return found

    attn = ['query', 'key', 'value', 'qkv', 'q_proj', 'k_proj', 'v_proj', 'proj', 'out_proj']
    found = [n for n in all_names if n in attn]
    if found:
        return found

    print(f'  WARNING: No attention layers found. Using all: {all_names}')
    return list(all_names)


print('Backbone loader ready.')

In [ ]:
# ============================================================
# Cell 7: Multi-Task Model with Shared Backbone + Per-Task Heads
# ============================================================
from peft import LoraConfig, get_peft_model


class MultiTaskFineTuneModel(nn.Module):
    """
    Shared LoRA-adapted backbone → per-task classification heads.

    Architecture:
        Image → [Shared Backbone + LoRA] → embedding (D-dim)
                                            ├→ Head_DR → logit_DR
                                            ├→ Head_Glaucoma → logit_Glaucoma
                                            ├→ Head_Edema → logit_Edema
                                            └→ Head_... → logit_...

    The embedding is what gets extracted after training — shared across all tasks.
    """

    def __init__(self, backbone, backbone_name, embed_dim, tasks):
        super().__init__()
        self.backbone = backbone
        self.backbone_name = backbone_name
        self.embed_dim = embed_dim
        self.tasks = tasks

        # Shared feature normalization
        self.shared_norm = nn.LayerNorm(embed_dim)
        self.shared_dropout = nn.Dropout(0.2)

        # Per-task classification heads
        # Each head is lightweight: Linear → GELU → Dropout → Linear → 1
        self.heads = nn.ModuleDict()
        for task in tasks:
            self.heads[task] = nn.Sequential(
                nn.Linear(embed_dim, 128),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(128, 1),
            )

    def forward_features(self, x):
        """Extract shared embedding from backbone (before task heads)."""
        if self.backbone_name == 'convnextv2_base':
            out = self.backbone(x)
            return out.pooler_output

        elif self.backbone_name.startswith('dinov3'):
            out = self.backbone(pixel_values=x)
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                return out.pooler_output
            return out.last_hidden_state[:, 0, :]

        elif self.backbone_name == 'vit_base':
            out = self.backbone(pixel_values=x)
            return out.pooler_output

        elif 'RETFound_mae' in self.backbone_name:
            return self.backbone.forward_features(x)

        elif 'RETFound_dinov2' in self.backbone_name:
            latent = self.backbone.forward_features(x)
            return latent[:, 1:, :].mean(dim=1)

        else:
            raise ValueError(f'Unknown backbone: {self.backbone_name}')

    def forward(self, x):
        """
        Returns dict of logits: {task_name: (B,) tensor}.
        """
        features = self.forward_features(x)       # (B, D)
        features = self.shared_norm(features)       # Normalize
        features = self.shared_dropout(features)    # Dropout (training only)

        logits = {}
        for task in self.tasks:
            logits[task] = self.heads[task](features).squeeze(-1)  # (B,)

        return logits


def build_multitask_model(backbone_name):
    """Load backbone → apply LoRA → wrap with multi-task heads."""
    print(f'\n{"="*60}')
    print(f'Building: {backbone_name}')
    print(f'{"="*60}')

    backbone, embed_dim, target_modules = load_backbone(backbone_name)

    total_params = sum(p.numel() for p in backbone.parameters())
    print(f'  Total backbone params: {total_params:,}')
    print(f'  Embed dim: {embed_dim}')
    print(f'  LoRA targets: {target_modules}')

    # Apply LoRA
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias='none',
    )
    backbone = get_peft_model(backbone, lora_config)

    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f'  LoRA trainable: {trainable:,} ({100*trainable/total_params:.2f}%)')

    # Build multi-task model
    model = MultiTaskFineTuneModel(backbone, backbone_name, embed_dim, TASKS)

    head_params = sum(p.numel() for p in model.heads.parameters())
    norm_params = sum(p.numel() for p in model.shared_norm.parameters())
    print(f'  Task heads ({len(TASKS)} tasks): {head_params:,} params')
    print(f'  Total trainable: {trainable + head_params + norm_params:,}')

    return model.to(device), embed_dim


print('Multi-task model builder ready.')

In [ ]:
# ============================================================
# Cell 8: Multi-Task Loss Function
# ============================================================


class MultiTaskBCELoss(nn.Module):
    """
    Combined BCE loss across all tasks, handling missing labels.

    For each task:
      loss_task = BCE(logit, label) * mask * task_weight * pos_weight

    Where mask=0 for samples with missing labels → their loss is zero.
    Total loss = sum of per-task losses / number of active tasks.
    """

    def __init__(self, tasks, pos_weights_dict, task_loss_weights):
        super().__init__()
        self.tasks = tasks

        # Per-task BCE with class imbalance correction
        self.criteria = {}
        for task in tasks:
            pw = torch.tensor([pos_weights_dict[task]], dtype=torch.float32).to(device)
            self.criteria[task] = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')

        # Task importance weights
        self.task_weights = task_loss_weights

    def forward(self, logits_dict, labels, masks):
        """
        Args:
            logits_dict: {task_name: (B,) logits}
            labels: (B, num_tasks) tensor
            masks:  (B, num_tasks) tensor (1 = valid, 0 = missing)
        """
        total_loss = 0.0
        active_tasks = 0
        per_task_losses = {}

        for i, task in enumerate(self.tasks):
            task_labels = labels[:, i]  # (B,)
            task_masks = masks[:, i]    # (B,)
            task_logits = logits_dict[task]  # (B,)

            if task_masks.sum() == 0:
                # No valid labels for this task in this batch
                per_task_losses[task] = 0.0
                continue

            # Per-sample loss, masked
            raw_loss = self.criteria[task](task_logits, task_labels)  # (B,)
            masked_loss = (raw_loss * task_masks).sum() / task_masks.sum()

            # Apply task importance weight
            weighted_loss = masked_loss * self.task_weights.get(task, 1.0)

            total_loss += weighted_loss
            active_tasks += 1
            per_task_losses[task] = masked_loss.item()

        # Average across active tasks
        if active_tasks > 0:
            total_loss = total_loss / active_tasks

        return total_loss, per_task_losses


print('Multi-task loss ready.')

In [ ]:
# ============================================================
# Cell 9: Training & Evaluation Functions
# ============================================================
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm


class EarlyStopping:
    """Early stopping on mean validation F1 across all tasks."""
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = -1
        self.counter = 0
        self.best_state = None

    def __call__(self, score, model):
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0
    n_batches = 0
    # Collect per-task predictions
    all_preds = {task: [] for task in TASKS}
    all_labels = {task: [] for task in TASKS}
    all_masks = {task: [] for task in TASKS}

    for images, labels_batch, masks_batch, _ in tqdm(loader, desc='  Train', leave=False):
        images = images.to(device, non_blocking=True)
        labels_batch = labels_batch.to(device, non_blocking=True)
        masks_batch = masks_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast(dtype=torch.float16):
            logits_dict = model(images)
            loss, _ = criterion(logits_dict, labels_batch, masks_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

        # Collect predictions per task
        for i, task in enumerate(TASKS):
            mask = masks_batch[:, i].cpu().numpy().astype(bool)
            if mask.sum() > 0:
                preds = torch.sigmoid(logits_dict[task]).detach().cpu().numpy()[mask]
                labs = labels_batch[:, i].cpu().numpy()[mask]
                all_preds[task].extend(preds)
                all_labels[task].extend(labs)

    avg_loss = total_loss / max(n_batches, 1)

    # Per-task metrics
    task_f1s = {}
    for task in TASKS:
        if len(all_preds[task]) > 0:
            p = np.array(all_preds[task])
            l = np.array(all_labels[task])
            task_f1s[task] = f1_score(l, (p >= 0.5).astype(int), zero_division=0)
        else:
            task_f1s[task] = 0.0

    return avg_loss, task_f1s


@torch.no_grad()
def evaluate_multitask(model, loader):
    model.eval()
    all_preds = {task: [] for task in TASKS}
    all_labels = {task: [] for task in TASKS}

    for images, labels_batch, masks_batch, _ in tqdm(loader, desc='  Eval', leave=False):
        images = images.to(device, non_blocking=True)
        labels_batch = labels_batch.to(device, non_blocking=True)
        masks_batch = masks_batch.to(device, non_blocking=True)

        with autocast(dtype=torch.float16):
            logits_dict = model(images)

        for i, task in enumerate(TASKS):
            mask = masks_batch[:, i].cpu().numpy().astype(bool)
            if mask.sum() > 0:
                preds = torch.sigmoid(logits_dict[task]).cpu().numpy()[mask]
                labs = labels_batch[:, i].cpu().numpy()[mask]
                all_preds[task].extend(preds)
                all_labels[task].extend(labs)

    # Compute per-task metrics
    results = {}
    for task in TASKS:
        p = np.array(all_preds[task])
        l = np.array(all_labels[task])
        if len(p) > 0 and len(np.unique(l)) > 1:
            pred_bin = (p >= 0.5).astype(int)
            results[task] = {
                'f1': f1_score(l, pred_bin, zero_division=0),
                'auc': roc_auc_score(l, p),
                'precision': precision_score(l, pred_bin, zero_division=0),
                'recall': recall_score(l, pred_bin, zero_division=0),
                'n_samples': len(l),
            }
        else:
            results[task] = {'f1': 0, 'auc': 0, 'precision': 0, 'recall': 0, 'n_samples': len(l)}

    # Mean F1 across primary tasks (DR + glaucoma) for early stopping
    primary_tasks = ['diabetic_retinopathy', 'increased_cup_disc']
    mean_f1 = np.mean([results[t]['f1'] for t in primary_tasks if t in results])

    return results, mean_f1


print('Training functions ready.')

In [ ]:
# ============================================================
# Cell 10: Main Multi-Task Training Loop
# ============================================================

def train_multitask(backbone_name):
    """
    Full multi-task fine-tuning pipeline for one backbone.
    Returns (model, test_results, embed_dim, history).
    """
    model, embed_dim = build_multitask_model(backbone_name)

    # Separate param groups
    backbone_params = [p for n, p in model.named_parameters()
                       if 'heads' not in n and 'shared_' not in n and p.requires_grad]
    head_params = list(model.heads.parameters()) + list(model.shared_norm.parameters())

    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params, 'lr': LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-7
    )

    criterion = MultiTaskBCELoss(TASKS, pos_weights, TASK_LOSS_WEIGHTS)
    scaler = GradScaler()
    early_stop = EarlyStopping(patience=PATIENCE)

    print(f'\nTraining {backbone_name} — {len(TASKS)} tasks — {EPOCHS} epochs')
    print(f'  LR backbone: {LR_BACKBONE}, LR heads: {LR_HEAD}')
    print()

    history = []
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_f1s = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        scheduler.step()

        val_results, val_mean_f1 = evaluate_multitask(model, val_loader)

        # Log
        dr_f1 = val_results['diabetic_retinopathy']['f1']
        gl_f1 = val_results['increased_cup_disc']['f1']
        ed_f1 = val_results['macular_edema']['f1']

        epoch_row = {
            'epoch': epoch, 'train_loss': train_loss,
            'val_mean_f1': val_mean_f1,
            'val_dr_f1': dr_f1, 'val_glaucoma_f1': gl_f1, 'val_edema_f1': ed_f1,
        }
        for task in TASKS:
            epoch_row[f'val_{task}_f1'] = val_results[task]['f1']
            epoch_row[f'val_{task}_auc'] = val_results[task]['auc']
        history.append(epoch_row)

        print(f'  Epoch {epoch:2d}/{EPOCHS} | Loss: {train_loss:.4f} | '
              f'Val mean_F1: {val_mean_f1:.3f} | '
              f'DR_F1: {dr_f1:.3f} Glauc_F1: {gl_f1:.3f} Edema_F1: {ed_f1:.3f}')

        if early_stop(val_mean_f1, model):
            print(f'  Early stopping at epoch {epoch}. Best mean F1: {early_stop.best_score:.3f}')
            break

    # Restore best
    if early_stop.best_state is not None:
        model.load_state_dict(early_stop.best_state)
        print(f'  Restored best checkpoint (mean F1 = {early_stop.best_score:.3f})')

    # Test
    print(f'\n  === Test Results: {backbone_name} ===')
    test_results, test_mean_f1 = evaluate_multitask(model, test_loader)
    print(f'  {"Task":30s} {"F1":>8} {"AUC":>8} {"Prec":>8} {"Recall":>8} {"N":>6}')
    print(f'  {"-"*72}')
    for task in TASKS:
        r = test_results[task]
        print(f'  {task:30s} {r["f1"]:8.3f} {r["auc"]:8.3f} {r["precision"]:8.3f} {r["recall"]:8.3f} {r["n_samples"]:6d}')
    print(f'  {"-"*72}')
    print(f'  {"MEAN (DR+Glauc)":30s} {test_mean_f1:8.3f}')

    return model, test_results, embed_dim, history


print('Training pipeline ready.')

In [ ]:
# ============================================================
# Cell 11: Embedding Extraction
# ============================================================

@torch.no_grad()
def extract_embeddings(model, loader):
    """
    Extract embeddings from the shared backbone (before task heads).
    These are the GENERAL-PURPOSE fine-tuned embeddings.
    """
    model.eval()
    all_names = []
    all_features = []

    for images, _, _, filenames in tqdm(loader, desc='Extracting embeddings'):
        images = images.to(device, non_blocking=True)

        with autocast(dtype=torch.float16):
            features = model.forward_features(images)

        all_features.append(features.float().cpu().numpy())
        all_names.extend(filenames)

    features_np = np.concatenate(all_features, axis=0)
    print(f'Embeddings shape: {features_np.shape}')

    # Same format as frozen embeddings (feature_0, feature_1, ...)
    feature_cols = [f'feature_{i}' for i in range(features_np.shape[1])]
    df = pd.DataFrame(features_np, columns=feature_cols)
    df.insert(0, 'name', all_names)

    return df


print('Embedding extraction ready.')

---

## Run: Fine-Tune ALL 7 Backbones (Batch Mode)

Loops through all backbones. Saves results after each one.
**Skips already-completed backbones** — safe to restart if Colab disconnects.

In [ ]:
# ============================================================
# Cell 12: Batch Fine-Tuning — ALL 7 Backbones
# ============================================================
import gc

ALL_BACKBONES = [
    'convnextv2_base',
    'dinov3_convnext_base',
    'dinov3_vitb16',
    'RETFound_dinov2_shanghai',
    'RETFound_mae_natureCFP',
    'RETFound_mae_shanghai',
    'vit_base',
]

all_results = []

for backbone_name in ALL_BACKBONES:
    out_filename = f'Embeddings_brset_finetuned_multitask_{backbone_name}.csv'
    out_path = os.path.join(OUTPUT_DIR, out_filename)

    if os.path.exists(out_path):
        print(f'\nSKIPPING {backbone_name} — already done: {out_path}')
        continue

    try:
        # Fine-tune
        model_ft, test_res, edim, hist = train_multitask(backbone_name)

        # Extract embeddings from ALL images
        emb_df = extract_embeddings(model_ft, full_loader)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        emb_df.to_csv(out_path, index=False)
        print(f'  Saved embeddings: {out_path}')

        # Save training history
        h_path = os.path.join(OUTPUT_DIR, f'history_multitask_{backbone_name}.csv')
        pd.DataFrame(hist).to_csv(h_path, index=False)

        # Save LoRA weights
        lora_dir = os.path.join(OUTPUT_DIR, f'lora_multitask_{backbone_name}')
        model_ft.backbone.save_pretrained(lora_dir)

        # Collect per-task test results
        row = {'backbone': backbone_name}
        for task in TASKS:
            row[f'{task}_f1'] = test_res[task]['f1']
            row[f'{task}_auc'] = test_res[task]['auc']
        all_results.append(row)

    except Exception as e:
        print(f'\n  ERROR with {backbone_name}: {e}')
        import traceback
        traceback.print_exc()
        all_results.append({'backbone': backbone_name, 'error': str(e)})

    finally:
        # Free GPU for next backbone
        if 'model_ft' in dir():
            del model_ft
        torch.cuda.empty_cache()
        gc.collect()

# Save grand summary
if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_path = os.path.join(OUTPUT_DIR, 'multitask_finetuning_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f'\n{"="*60}')
    print('ALL RESULTS — Multi-Task Fine-Tuning')
    print(f'{"="*60}')
    # Show key metrics
    display_cols = ['backbone'] + [f'{t}_f1' for t in ['diabetic_retinopathy', 'increased_cup_disc', 'macular_edema']]
    display_cols = [c for c in display_cols if c in summary_df.columns]
    print(summary_df[display_cols].to_string(index=False))
    print(f'\nFull summary: {summary_path}')

## Results: Frozen vs Fine-Tuned Comparison

Quick LogReg evaluation on both frozen and multi-task fine-tuned embeddings.
Shows how much the embeddings improved **for each task**.

In [ ]:
# ============================================================
# Cell 13: Frozen vs Fine-Tuned Comparison
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def eval_embeddings_logreg(emb_csv, labels_df, test_patient_ids, task_col):
    """Quick LogReg eval on an embedding CSV. Returns (f1, auc)."""
    emb = pd.read_csv(emb_csv)
    id_col = 'name' if 'name' in emb.columns else 'ImageName'
    feat_cols = [c for c in emb.columns if c != id_col]

    emb['_key'] = emb[id_col].astype(str).str.replace(r'\.jpg$', '', regex=True)
    lbl = labels_df.copy()
    lbl['_key'] = lbl['image_file'].astype(str).str.replace(r'\.jpg$', '', regex=True)

    m = emb.merge(lbl[['_key', 'patient_id', task_col]], on='_key', how='inner')
    m = m.dropna(subset=[task_col])

    train_m = m[~m['patient_id'].isin(test_patient_ids)]
    test_m = m[m['patient_id'].isin(test_patient_ids)]

    if len(test_m) < 10 or len(train_m) < 10:
        return -1, -1

    X_tr = StandardScaler().fit_transform(train_m[feat_cols].values.astype(np.float32))
    y_tr = train_m[task_col].values.astype(int)

    sc = StandardScaler().fit(train_m[feat_cols].values.astype(np.float32))
    X_tr = sc.transform(train_m[feat_cols].values.astype(np.float32))
    X_te = sc.transform(test_m[feat_cols].values.astype(np.float32))
    y_te = test_m[task_col].values.astype(int)

    lr = LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0, random_state=42)
    lr.fit(X_tr, y_tr)

    return (
        f1_score(y_te, lr.predict(X_te), zero_division=0),
        roc_auc_score(y_te, lr.predict_proba(X_te)[:, 1])
    )


# --- Where are the frozen embeddings? (ADJUST THIS) ---
FROZEN_PATHS = {
    'convnextv2_base':          '/content/drive/MyDrive/PATH_TO/Embeddings_brset_convnextv2_base_.csv',
    'dinov3_convnext_base':     '/content/drive/MyDrive/PATH_TO/Embeddings_brset_dinov3_convnext_base.csv',
    'dinov3_vitb16':            '/content/drive/MyDrive/PATH_TO/Embeddings_brset_dinov3_vitb16.csv',
    'RETFound_dinov2_shanghai': '/content/drive/MyDrive/PATH_TO/Embeddings_brset_RETFound_dinov2_shanghai.csv',
    'RETFound_mae_natureCFP':   '/content/drive/MyDrive/PATH_TO/Embeddings_brset_RETFound_mae_natureCFP.csv',
    'RETFound_mae_shanghai':    '/content/drive/MyDrive/PATH_TO/Embeddings_brset_RETFound_mae_shanghai.csv',
    'vit_base':                 '/content/drive/MyDrive/PATH_TO/Embeddings_brset_vit_base_.csv',
}

# --- Compare for key tasks ---
COMPARE_TASKS = ['diabetic_retinopathy', 'increased_cup_disc', 'macular_edema', 'drusens']

print(f'{"":<30}', end='')
for task in COMPARE_TASKS:
    short = task[:12]
    print(f' | {short:>12} Frz  FT', end='')
print()
print('-' * 120)

comparison_rows = []
for backbone_name in ALL_BACKBONES:
    ft_path = os.path.join(OUTPUT_DIR, f'Embeddings_brset_finetuned_multitask_{backbone_name}.csv')
    frozen_path = FROZEN_PATHS.get(backbone_name, '')

    print(f'{backbone_name:<30}', end='')
    row = {'backbone': backbone_name}

    for task in COMPARE_TASKS:
        frz_f1, frz_auc = (-1, -1)
        ft_f1, ft_auc = (-1, -1)

        if os.path.exists(frozen_path):
            frz_f1, frz_auc = eval_embeddings_logreg(frozen_path, labels, test_pats, task)
        if os.path.exists(ft_path):
            ft_f1, ft_auc = eval_embeddings_logreg(ft_path, labels, test_pats, task)

        print(f' | {frz_f1:6.3f} {ft_f1:6.3f}', end='')
        row[f'{task}_frozen_f1'] = frz_f1
        row[f'{task}_ft_f1'] = ft_f1
        row[f'{task}_frozen_auc'] = frz_auc
        row[f'{task}_ft_auc'] = ft_auc

    print()
    comparison_rows.append(row)

comp_df = pd.DataFrame(comparison_rows)
comp_path = os.path.join(OUTPUT_DIR, 'frozen_vs_multitask_finetuned.csv')
comp_df.to_csv(comp_path, index=False)
print(f'\nComparison saved to: {comp_path}')

## Using These Embeddings in Your Evaluation Notebooks

The fine-tuned embeddings are saved in the **same format** as frozen ones:

```
name, feature_0, feature_1, ..., feature_767
img00001.jpg, 0.123, -0.456, ..., 0.789
```

To use them in your existing evaluation notebooks:

1. Copy the CSV files from `finetuned_embeddings_multitask/` into `data/brset_embeddings/`
2. Your `load_retina_embeddings_dataset()` function will load them automatically
3. Run the same evaluation pipeline (LogReg, XGBoost, MLP) on both frozen and fine-tuned

### What tasks can these embeddings handle?

Because they were trained on **8 tasks simultaneously**, these embeddings are good for:
- DR detection (any grade)
- Glaucoma suspect detection
- Macular edema detection
- Referral to specialist (composite of above)
- AMD, drusens, hypertension, scarring detection
- *And potentially*: diabetes detection, comorbidity prediction (transfer learning from shared features)